# Models: 2016-19, school variables only

- **Paper:** Fig 1(a), Fig 3A
- **Original notebook:** `24_pre_covid_school_only.ipynb`
- **Reads:**
  - `outputs/school_related_pre.pkl`
  - `data/raw/school_related/CEnroll2021.txt`
- **Writes:**
  - `outputs/24_pred_actual.pkl`
  - `outputs/24_shap_values.pkl`
  - `outputs/24_*_trial.pkl`


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup


import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import random
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import xgboost as xgb
#from sklearn.metrics import aic
import graphviz




from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

# pip install rioxarray  # see requirements.txt
# pip install pysal  # see requirements.txt
# pip install contextily  # see requirements.txt
import libpysal


import rioxarray  # surface data manipulation
import xarray  # surface data manipulation
from pysal.explore import esda  # spatial stats
from pysal.lib import weights  # weights
import contextily



In [ ]:
# pip install optuna  # see requirements.txt
import optuna

# pip install shap  # see requirements.txt
import shap

In [ ]:
# set seeds

SEED = 123

random.seed(SEED)

np.random.seed(SEED)

from optuna.samplers import TPESampler
sampler = TPESampler(seed=SEED)

In [ ]:
# load data
# import data


# cumulative enrollment
total_enrolled = pd.read_csv(f"{DATA_DIR}/school_related/CEnroll2021.txt",sep="\t",encoding='latin-1')

school_related = pd.read_pickle(f"{OUTPUT_DIR}/school_related_pre.pkl")

relevant  = school_related.drop(columns = ['SchoolCode','SchoolName','CountyName',"DistrictCode",'DistrictName','closest_UC','closest_CSU'])
# relevant = school_related[['college_going_rate', 'YearsTeaching', 'years_teaching_std',
#        'ChronicAbsenteeismRate', 'pct_frpm', 'pct_passed_science',
#        'college_going_rate', 'pct_passed_english', 'school_expenditures',
#        'ShannonIndex', 'ProportionMale', 'ProportionWhite', 'DistrictCode']]

relevant
for col in relevant.columns:
    school_related[col] = pd.to_numeric(school_related[col], errors='coerce')



# cleaning
cleaned_data = school_related.dropna(subset=relevant.columns.values)


In [ ]:
# getting diversity, pwhite, and pmale
"""
RB = African American
RI = American Indian or Alaska Native
RA = Asian
RF = Filipino
RH = Hispanic or Latino
RD = Not Reported
RP = Pacific Islander
RT = Two or More Races
RW = White
"""

total_enrolled = total_enrolled[total_enrolled['AggregateLevel']=='S']

# group by school
# race categories
cats = ['RB', 'RI', 'RA', 'RF', 'RH', 'RD', 'RP', 'RT', 'RW','GM']
total_enrolled['CumulativeEnrollment'] = pd.to_numeric(total_enrolled['CumulativeEnrollment'],errors='coerce')
et_enrolled = total_enrolled[total_enrolled['ReportingCategory'].isin(cats)]

# by school
grouped = et_enrolled.groupby('SchoolName')
shannon_indices = []

# shannon for each school
for school, data in grouped:
    data['CumulativeEnrollment'].fillna(0, inplace=True)
    total_male_students = data[data['ReportingCategory'] == 'GM']['CumulativeEnrollment'].sum()
    data = data[data['ReportingCategory'] != 'GM']
    total_students = data['CumulativeEnrollment'].sum()

    school_code = data['SchoolCode'].iloc[0]
    district_code = data['DistrictCode'].iloc[0]
    data['CumulativeEnrollment'].fillna(0, inplace=True)

    # proportion of students in each category
    proportions = data['CumulativeEnrollment'] / total_students

    # shannon entropy
    shannon_entropy = np.exp(-np.sum(proportions * np.log(proportions)))

    # proportion of students
    proportion_white = data.loc[data['ReportingCategory'] == 'RW', 'CumulativeEnrollment'].sum() / total_students

    # proportion of male students
    proportion_male = total_male_students / total_students
    # append
    shannon_indices.append({'SchoolName': school, 'ShannonIndex': shannon_entropy,
                             'ProportionMale': proportion_male, 'ProportionWhite':proportion_white, 'SchoolCode': school_code,
                             'DistrictCode': district_code})



shannon_df = pd.DataFrame(shannon_indices)



In [ ]:
shannon_df
school_merged = pd.merge(school_related, shannon_df,on = ['SchoolCode','SchoolName'],how = 'left')
# school_merged
# with open(f'{OUTPUT_DIR}/all_schoolwide_vars.pkl','wb') as f:
#   pickle.dump(school_merged,f)


In [ ]:
# getting relevant columns



# relevant.rename(columns = {'SchoolName_x':"SchoolName", 'Percent (%) \nEligible FRPM \n(K-12)':'percent_frpm','College Going Rate - Total (12 Months)':'college_rate'},inplace=True)
# relevant = school_merged.drop(columns = ['SchoolCode','SchoolName',"CountyName",'n_teachers','DistrictCode','YearsTeaching','BeyondBachelors','TeachingExp_2'])
# relevant  = school_merged.drop(columns = ['SchoolCode','SchoolName',"Latitude",'Longitude'])#,"IB_Count"])

school_merged


In [ ]:

from sklearn.model_selection import train_test_split




relevant = school_merged[['CountyName',  'AP_Count', 'Latitude','Longitude',"pct_passed_math",
                     'YearsTeaching', 'years_teaching_std', 'distance_to_uc', 'distance_to_csu',
                     'ChronicAbsenteeismRate', 'pct_frpm', 'college_going_rate',
                     'school_expenditures', 'ProportionWhite', 'ProportionMale', 'ShannonIndex']]


relevant.dropna(inplace=True)


def split_by_county(df, test_size=0.2, seed=123):
    np.random.seed(seed)
    counties = df['CountyName'].unique()
    train_list = []
    test_list = []
    small_counties = []

    for county in counties:
        county_data = df[df['CountyName'] == county]
        if len(county_data) < 5:
            small_counties.append(county_data)
        else:
            county_train, county_test = train_test_split(county_data, test_size=.12,train_size = .477, random_state=seed)
            train_list.append(county_train)
            test_list.append(county_test)

    if small_counties:
        small_counties_df = pd.concat(small_counties)
        small_train, small_test = train_test_split(small_counties_df, test_size=.1,train_size = .47, random_state=seed)
        train_list.append(small_train)
        test_list.append(small_test)

    train_df = pd.concat(train_list)
    test_df = pd.concat(test_list)
    return train_df, test_df

# split by counties
train_df, test_df = split_by_county(relevant)

with open(f'{OUTPUT_DIR}/stratified_split_train.pkl','wb') as f:
  pickle.dump(train_df,f)


with open(f'{OUTPUT_DIR}/stratified_split_test.pkl','wb') as f:
  pickle.dump(test_df,f)



# x and y
X_train = train_df.drop(columns=['college_going_rate', 'CountyName','Latitude','Longitude',"pct_passed_math"])
y_train = train_df['college_going_rate']

X_test = test_df.drop(columns=[ 'college_going_rate', 'CountyName','Latitude','Longitude',"pct_passed_math"])
y_test = test_df['college_going_rate']

X_all = school_merged

# z-score normalization
X_train_zscore = (X_train - np.mean(X_train, axis=0)) / np.std(X_train, axis=0)
X_test_zscore = (X_test - np.mean(X_test, axis=0)) / np.std(X_test, axis=0)

print('n row X train:', X_train_zscore.shape[0])
print('n row X test:', X_test_zscore.shape[0])

features = ["Number AP Classes","Years Teaching (mean)","Years Teadching (SD)", "Distance to Closest UC",
            "Distance to Closest CSU","Chronic Absenteeism Rate","Free and Reduced Price Meals Rate","Per Pupil Expenditures",
            "Proportion White","Proportion Male","Shannon Index"]

XGBoost

In [ ]:
plt.rc('axes.spines', **{'bottom': True, 'left': True, 'right': True, 'top': True})

from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial):
  # set range for hyperparams
  n_estimators = trial.suggest_int('n_estimators',10,200)
  learning_rate = trial.suggest_float('learning_rate',0.001,0.1,log=True)
  max_depth= trial.suggest_int('max_depth',1,10)
  subsample = trial.suggest_float('subsample',0.05,1)
  colsample_bytree = trial.suggest_float('colsample_bytree',0.05,1.0)
  min_child_weight = trial.suggest_int('min_child_weight',1,20)

  # define model


  model = xgb.XGBRegressor(n_estimators = n_estimators,
                                max_depth = max_depth,
                                learning_rate = learning_rate,
                                subsample = subsample,
                                colsample_bytree = colsample_bytree,
                                min_child_weight = min_child_weight)

  # model.fit(X_train, y_train)
  # y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()
#   r2 = r2_score(y_test, y_pred)
#   rmse = mean_squared_error(y_test, y_pred,squared = False)

  return score_r2 ,score_rmse

if os.path.exists(f'{OUTPUT_DIR}/24_xgb_trial.pkl1'):
    study_xgb = pd.read_pickle(f'{OUTPUT_DIR}/24_xgb_trial.pkl')
else:
    study_xgb = optuna.create_study(directions = ['maximize','minimize'],sampler=sampler)
    study_xgb.optimize(objective_xgb, n_trials=100,show_progress_bar = True)
    with open(f'{OUTPUT_DIR}/24_xgb_trial.pkl',"wb") as f:
        pickle.dump(study_xgb,f)
xgb_best = study_xgb.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_xgb, target_names=["r2", "rmse"])


In [ ]:
print(f"Number of trials on the Pareto front: {len(study_xgb.best_trials)}")

trial_with_highest_accuracy_xgb = max(study_xgb.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_xgb.number}")
print(f"\tparams: {trial_with_highest_accuracy_xgb.params}")
print(f"\tvalues: {trial_with_highest_accuracy_xgb.values}")

In [ ]:
# XGboost


# train
n = 100
xgb_model = xgb.XGBRegressor(**trial_with_highest_accuracy_xgb.params)
xgb_model.fit(X_train,y_train,verbose = False)


preds = xgb_model.predict(X_test)
rmse_xgb = root_mean_squared_error(y_test, preds)
r2_xgb = r2_score(y_test, preds)
print("RMSE: ",rmse_xgb)
print("R-squared:", r2_xgb)
# show r^2
# xgb.plot_importance(model)

feature_importances_xgb = xgb_model.feature_importances_

feature_importance_dict_xgb = dict(zip(X_train.columns, feature_importances_xgb))
feature_importance_xgb = pd.DataFrame(feature_importance_dict_xgb.items(), columns=['Feature', 'Importance'])
feature_importance_xgb["Importance"] =feature_importance_xgb["Importance"]/feature_importance_xgb["Importance"].sum()

preds_xgb = preds


In [ ]:
# shap xgb
explainer_xgb = shap.Explainer(xgb_model)
shap_values_xgb = explainer_xgb.shap_values(X_test)
shap.summary_plot(shap_values_xgb, X_test,feature_names = features)

In [ ]:
shap.summary_plot(shap_values_xgb, X_test,feature_names = features,plot_type = 'bar')

random forest regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor


def objective_rf(trial):
  # set range for hyperparams
  n_estimators = trial.suggest_int('n_estimators',10,200)
  max_depth= trial.suggest_int('max_depth',1,10)
  min_samples_split = trial.suggest_int('min_samples_split',2,32)
  min_samples_leaf = trial.suggest_int('min_samples_leaf',2,32)

  # define model


  model = RandomForestRegressor(n_estimators = n_estimators,
                                max_depth = max_depth,
                                min_samples_split = min_samples_split,
                                min_samples_leaf = min_samples_leaf)

  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()
  # r2 = r2_score(y_test, y_pred)
  # rmse = mean_squared_error(y_test, y_pred,squared = False)

  return score_r2 ,score_rmse

if os.path.exists(f'{OUTPUT_DIR}/24_rf_trial.pkl1'):
    study_rf = pd.read_pickle(f'{OUTPUT_DIR}/24_rf_trial.pkl')
else:
    study_rf = optuna.create_study(directions = ['maximize','minimize'],sampler=sampler)

    study_rf.optimize(objective_rf, n_trials=100,show_progress_bar = True)
    with open(f'{OUTPUT_DIR}/24_rf_trial.pkl',"wb") as f:
        pickle.dump(study_rf,f)


rf_best = study_rf.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_rf, target_names=["r2", "rmse"])

In [ ]:
print(f"Number of trials on the Pareto front: {len(study_rf.best_trials)}")

trial_with_highest_accuracy_rf = max(study_rf.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_rf.number}")
print(f"\tparams: {trial_with_highest_accuracy_rf.params}")
print(f"\tvalues: {trial_with_highest_accuracy_rf.values}")

In [ ]:
rfr = RandomForestRegressor(**trial_with_highest_accuracy_rf.params)

rfr.fit(X_train, y_train)
y_pred = rfr.predict(X_test)
rmse_rf = root_mean_squared_error(y_test, y_pred)
r2_rf = r2_score(y_test, y_pred)
print("RMSE: ",rmse_rf)
print("R-squared:", r2_rf)
forest_importances = pd.Series(rfr.feature_importances_, index=rfr.feature_names_in_)
std = np.std([tree.feature_importances_ for tree in rfr.estimators_], axis=0)

# feature importance based on feature permutation
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rfr, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2
)

forest_importances = pd.Series(result.importances_mean, index=rfr.feature_names_in_)
forest_importances = forest_importances/forest_importances.sum()

preds_rf = y_pred

In [ ]:
# shap rf
explainer_rf = shap.Explainer(rfr)
shap_values_rf = explainer_rf.shap_values(X_test)
shap.summary_plot(shap_values_rf, X_test,feature_names = features)

In [ ]:
shap.summary_plot(shap_values_rf, X_test,feature_names = features,plot_type='bar')

# Lasso

In [ ]:
from sklearn.linear_model import Lasso

def objective_lasso(trial):
  # set range for hyperparams
  alpha = trial.suggest_float('alpha',0,1)
  # define model


  model = Lasso(alpha = alpha)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()
  # r2 = r2_score(y_test, y_pred)
  # rmse = mean_squared_error(y_test, y_pred,squared = False)

  return score_r2 ,score_rmse

if os.path.exists(f'{OUTPUT_DIR}/24_lasso_trial.pkl1'):
    study_lasso = pd.read_pickle(f'{OUTPUT_DIR}/24_lasso_trial.pkl')
else:

    study_lasso = optuna.create_study(directions = ['maximize','minimize'])

    study_lasso.optimize(objective_lasso, n_trials=100,show_progress_bar = True)
    with open(f'{OUTPUT_DIR}/24_lasso_trial.pkl',"wb") as f:
        pickle.dump(study_lasso,f)


lasso_best = study_lasso.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_lasso, target_names=["r2", "rmse"])

In [ ]:
print(f"Number of trials on the Pareto front: {len(study_lasso.best_trials)}")

trial_with_highest_accuracy_lasso = max(study_lasso.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_lasso.number}")
print(f"\tparams: {trial_with_highest_accuracy_lasso.params}")
print(f"\tvalues: {trial_with_highest_accuracy_lasso.values}")

In [ ]:

lasso = Lasso(**trial_with_highest_accuracy_lasso.params)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
rmse_lasso = root_mean_squared_error(y_test,y_pred_lasso)
r2_lasso = r2_score(y_test,y_pred_lasso)
lasso_coeff = pd.DataFrame()
lasso_coeff["Columns"] = X_train.columns
lasso_coeff['Coefficient Estimate'] = abs(pd.Series(lasso.coef_))
lasso_coeff['Coefficient Estimate'] = lasso_coeff['Coefficient Estimate']/lasso_coeff['Coefficient Estimate'].sum()

print("RMSE:",rmse_lasso)
print("R-squared:",r2_lasso)

preds_lasso = y_pred_lasso

In [ ]:
# shap lasso
explainer_lasso = shap.Explainer(lasso.predict,X_test)
shap_values_lasso = explainer_lasso.shap_values(X_test)
shap.summary_plot(shap_values_lasso, X_test,feature_names = features)

In [ ]:
shap.summary_plot(shap_values_lasso, X_test,feature_names = features,plot_type = 'bar')

In [ ]:
# pip install catboost  # see requirements.txt
from catboost import CatBoostRegressor

In [ ]:

def objective_cat(trial):
  # set range for hyperparams
  iterations = trial.suggest_int('iterations',10,200)
  learning_rate = trial.suggest_float('learning_rate',0.001,0.1,log=True)
  depth= trial.suggest_int('depth',1,10)
  subsample = trial.suggest_float('subsample',0.05,1)
  colsample_bylevel = trial.suggest_float('colsample_bylevel',0.05,1.0)
  min_data_in_leaf = trial.suggest_int('min_data_in_leaf',1,20)

  # define model


  model = CatBoostRegressor(iterations = iterations,
                                depth = depth,
                                learning_rate = learning_rate,
                                subsample = subsample,
                                colsample_bylevel = colsample_bylevel,
                                min_data_in_leaf = min_data_in_leaf,
                            verbose = False)

  # model.fit(X_train, y_train)
  # y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()


  return score_r2 ,score_rmse


if os.path.exists(f'{OUTPUT_DIR}/24_cat_trial.pkl1'):
    study_cat = pd.read_pickle(f'{OUTPUT_DIR}/24_cat_trial.pkl')
else:

    study_cat = optuna.create_study(directions = ['maximize','minimize'],sampler=sampler)

    study_cat.optimize(objective_cat, n_trials=100,show_progress_bar = True)
    with open(f'{OUTPUT_DIR}/24_cat_trial.pkl',"wb") as f:
        pickle.dump(study_cat,f)


cat_best = study_cat.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_cat, target_names=["r2", "rmse"])

In [ ]:
print(f"Number of trials on the Pareto front: {len(study_cat.best_trials)}")

trial_with_highest_accuracy_cat = max(study_cat.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_cat.number}")
print(f"\tparams: {trial_with_highest_accuracy_cat.params}")
print(f"\tvalues: {trial_with_highest_accuracy_cat.values}")

In [ ]:
catboost_model = CatBoostRegressor(**trial_with_highest_accuracy_cat.params)

catboost_model.fit(X_train, y_train, verbose= 0)

y_pred_cat = catboost_model.predict(X_test)
r2_cat = r2_score(y_test, y_pred_cat)

rmse_cat = root_mean_squared_error(y_test, y_pred_cat)
print("RMSE:", rmse_cat)
print("R-squared:",r2_cat)

feature_importance = catboost_model.get_feature_importance(type="PredictionValuesChange")

# feature importance
feature_names = X_train.columns
feature_importance_df = pd.DataFrame(list(zip(feature_names, feature_importance)),
                                      columns=['Feature', 'Importance'])
feature_importance_df['Importance'] = feature_importance_df['Importance']/feature_importance_df['Importance'].sum()

# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
# print(feature_importance_df)

preds_cat = y_pred_cat



In [ ]:
# shap cat
explainer_cat = shap.Explainer(catboost_model)
shap_values_cat = explainer_cat.shap_values(X_test)
shap.summary_plot(shap_values_cat, X_test,feature_names = features)

In [ ]:
shap.summary_plot(shap_values_cat, X_test,feature_names = features,plot_type = 'bar')

In [ ]:
shap.decision_plot(explainer_cat.expected_value, shap_values_cat, feature_names = features)

In [ ]:
shap.plots.violin(shap_values_cat,feature_names = features)


In [ ]:


# fig, axs = plt.subplots(2,2, figsize = (12,10))
# axs = axs.flatten()
# axs[0].barh(feature_importance_xgb['Feature'], feature_importance_xgb['Importance'])
# axs[0].set_xlabel('Importance')
# axs[0].set_ylabel('Coefficient Estimate')
# axs[0].set_title('XGBoost')

# axs[1].barh(forest_importances.index, forest_importances.values)
# axs[1].set_xlabel('Importance')
# axs[1].set_ylabel('Coefficient Estimate')
# axs[1].set_title('Random Forest')
# axs[1].set_yticklabels([])


# axs[2].barh(lasso_coeff['Columns'], lasso_coeff['Coefficient Estimate'])
# axs[2].set_xlabel('Importance')
# axs[2].set_ylabel('Coefficient Estimate')
# axs[2].set_title('Lasso')


# axs[3].barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
# axs[3].set_xlabel('Importance')
# axs[3].set_ylabel('Feature')
# axs[3].set_title('CatBoost')
# axs[3].set_yticklabels([])


# plt.show()
# # normalize values based on max for visualization

In [ ]:

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 14))
axes = axes.flatten()


plt.sca(axes[0])

shap.summary_plot(shap_values_xgb, X_test, sort = False
                  ,show=False,plot_type = "bar",feature_names = features)
axes[0].tick_params("both",labelsize =10)
axes[0].set_xlabel("mean |Shap value|",fontsize = 10)
axes[0].set_title("XGBoost")
axes[0].set_xlim(0,6)
plt.sca(axes[1])

shap.summary_plot(shap_values_rf, X_test, sort = False,
                  show=False,plot_type = "bar",feature_names = features)
axes[1].set_yticklabels([])
axes[1].tick_params("both",labelsize =10)

axes[1].set_title("Random Forest")
axes[1].set_xlabel("mean |Shap value|",fontsize = 10)
axes[1].set_xlim(0,6)

plt.sca(axes[2])

shap.summary_plot(shap_values_lasso, X_test, sort = False,
                  show=False,plot_type = "bar",feature_names = features)
axes[2].set_title("Lasso")
axes[2].set_xlabel("mean |Shap value|",fontsize = 10)
axes[2].tick_params("both",labelsize =10)
axes[2].set_xlim(0,6)

plt.sca(axes[3])

shap.summary_plot(shap_values_cat, X_test,sort = False,
                  show=False,plot_type = "bar",feature_names = features)
axes[3].set_yticklabels([])
axes[3].set_title("CatBoost")
axes[3].tick_params("both",labelsize =10)
axes[3].set_xlim(0,6)

axes[3].set_xlabel("mean |Shap value|",fontsize = 10)
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(30, 20))
axes = axes.flatten()


plt.sca(axes[0])

shap.summary_plot(shap_values_xgb, X_test, sort = False
                  ,show=False,feature_names = features,color_bar =False)
axes[0].tick_params("both",labelsize =10)
axes[0].set_xlabel("SHAP Value",fontsize = 10)
axes[0].set_title("XGBoost")

plt.sca(axes[1])

shap.summary_plot(shap_values_rf, X_test, sort = False,
                  show=False,feature_names = features)
axes[1].set_yticklabels([])
axes[1].tick_params("both",labelsize =10)

axes[1].set_title("Random Forest")
axes[1].set_xlabel("SHAP Value",fontsize = 10)
plt.sca(axes[2])

shap.summary_plot(shap_values_lasso, X_test, sort = False,
                  show=False,feature_names = features,color_bar =False)
axes[2].set_title("Lasso")
axes[2].set_xlabel("SHAP Value",fontsize = 10)
axes[2].tick_params("both",labelsize =10)

plt.sca(axes[3])

shap.summary_plot(shap_values_cat, X_test,sort = False,
                  show=False,feature_names = features)
axes[3].set_yticklabels([])
axes[3].set_title("CatBoost")
axes[3].tick_params("both",labelsize =10)

axes[3].set_xlabel("SHAP Value",fontsize = 10)
plt.tight_layout()
plt.show()


In [ ]:
# shap.plots.scatter(shap_values_xgb)


In [ ]:
pd.DataFrame({"model":['XGB','RF','Lasso','CatBoost'],'r-squared':[r2_xgb,r2_rf,r2_lasso,r2_cat],'rmse':[rmse_xgb,rmse_rf,rmse_lasso,rmse_cat]})


In [ ]:
preds = pd.DataFrame({"xgb":preds_xgb,"rf":preds_rf,"lasso":preds_lasso,"cat":preds_cat,"y_test":y_test})

plt.scatter(y_test, preds.iloc[:,0], alpha=0.5, label='XGBoost',s=20)
plt.scatter(y_test, preds.iloc[:,1], alpha=0.5, label='Random Forest',s=20)
plt.scatter(y_test, preds.iloc[:,2], alpha=0.5, label='Lasso',s=20)
plt.scatter(y_test, preds.iloc[:,3], alpha=0.5, label='CatBoost',s=20)

plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', label='y = x')

plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual Values')
plt.legend()


with open(f"{OUTPUT_DIR}/24_pred_actual.pkl","wb") as fi:
    pickle.dump(preds,fi)

In [ ]:
output = [shap_values_xgb,shap_values_rf,shap_values_lasso,shap_values_cat,X_test]


with open(f'{OUTPUT_DIR}/24_shap_values.pkl','wb') as f:
  pickle.dump(output,f)


expected = [
    np.mean(preds_xgb),
    np.mean(preds_rf),
    np.mean(preds_lasso),
    np.mean(preds_cat)
]

In [ ]:
X_all_f = X_all[['AP_Count', 'Latitude','Longitude','pct_passed_math',
                     'YearsTeaching', 'years_teaching_std', 'distance_to_uc', 'distance_to_csu',
                     'ChronicAbsenteeismRate', 'pct_frpm', 'college_going_rate',
                     'school_expenditures', 'ProportionWhite', 'ProportionMale', 'ShannonIndex']].dropna()

xgb_all = xgb_model.predict(X_all_f.drop(columns = ['college_going_rate','Latitude','Longitude','pct_passed_math']))
rf_all = rfr.predict(X_all_f.drop(columns = ['college_going_rate','Latitude','Longitude','pct_passed_math']))
lasso_all = lasso.predict(X_all_f.drop(columns = ['college_going_rate','Latitude','Longitude','pct_passed_math']))
cat_all = catboost_model.predict(X_all_f.drop(columns = ['college_going_rate','Latitude','Longitude','pct_passed_math']))


X_all_f['xgb_pred'] = xgb_all
X_all_f['rf_pred'] = rf_all
X_all_f['lasso_pred'] = lasso_all
X_all_f['cat_pred'] = cat_all

X_all_f['xgb_resid'] = xgb_all
X_all_f['rf_pred'] = rf_all
X_all_f['lasso_pred'] = lasso_all
X_all_f['cat_pred'] = cat_all

X_all_f

df_sp = gpd.GeoDataFrame(X_all_f,geometry=gpd.points_from_xy(X_all_f['Longitude'], X_all_f['Latitude']), crs='EPSG:4326')





In [ ]:

import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
df_sp['ensemble_pred'] = df_sp[['college_going_rate', 'xgb_pred', 'rf_pred', 'lasso_pred', 'cat_pred']].mean(axis= 1)

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 3, figsize=(10, 10))

axes[1][2].set_visible(False)

axes[1][0].set_position([0.24, 0.25, 0.228, 0.343])
axes[1][1].set_position([0.55, 0.25, 0.228, 0.343])
axes = axes.flatten()

norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.magma  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:5],
    ['college_going_rate', 'xgb_pred', 'rf_pred', 'lasso_pred', 'cat_pred'],
    ['Actual', 'XGB Prediction', 'RF Prediction', 'Lasso Prediction', 'Catboost Prediction']
):
    ca_counties.plot(ax=ax, color='lightgrey', edgecolor='white',linewidth = 0.5)

    df_sp.plot(column=col, ax=ax, markersize=1, cmap=cmap, norm=norm, legend=False)

    ax.set_axis_off()
    ax.set_title(title)

cax = fig.add_axes([0.91, 0.4, 0.015, 0.4])
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, cax=cax, orientation='vertical', label='College Going Rate (%)', ticks=[0, 25, 50, 75, 100])



plt.show()




In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(10, 10))


axes = axes.flatten()


norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.plasma

ca_counties.plot(ax=axes[0], color='lightgrey', edgecolor='white', linewidth=0.5)
df_sp.plot(column='college_going_rate', ax=axes[0], markersize=3, cmap=cmap, norm=norm, legend=False)
axes[0].set_title('Actual')

ca_counties.plot(ax=axes[1], color='lightgrey', edgecolor='white', linewidth=0.5)
df_sp.plot(column='ensemble_pred', ax=axes[1], markersize=3, cmap=cmap, norm=norm, legend=False)
axes[1].set_title('Ensemble Prediction')

axes[0].set_axis_off()
axes[1].set_axis_off()

cax = fig.add_axes([0.91, 0.35, 0.015, 0.35])

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, cax=cax, orientation='vertical', label='College Going Rate (%)', ticks=[0, 25, 50, 75, 100])

plt.subplots_adjust(left=0, right=0.85, top=0.95, bottom=0.05)
plt.show()

In [ ]:
from esda.moran import Moran_BV
from libpysal.weights import KNN

#knn weights
weights = KNN.from_dataframe(df_sp, k=12)
weights.transform = 'r'

print("Bivariate Moran's I:")
moran_bv_xgb = Moran_BV(df_sp['college_going_rate'], df_sp['xgb_pred'],weights)
print(f"(XGB): {moran_bv_xgb.I}, p-value: {moran_bv_xgb.p_sim}")

moran_bv_rf = Moran_BV(df_sp['college_going_rate'], df_sp['rf_pred'],weights)
print(f"(RF): {moran_bv_rf.I}, p-value: {moran_bv_rf.p_sim}")

moran_bv_lasso = Moran_BV(df_sp['college_going_rate'], df_sp['lasso_pred'],weights)
print(f"(Lasso): {moran_bv_lasso.I}, p-value: {moran_bv_lasso.p_sim}")

moran_bv_cat = Moran_BV(df_sp['college_going_rate'], df_sp['cat_pred'],weights)
print(f"(Catboost): {moran_bv_cat.I}, p-value: {moran_bv_lasso.p_sim}")


moran_bv_cat = Moran_BV(df_sp['college_going_rate'], df_sp['ensemble_pred'],weights)
print(f"(ensemble): {moran_bv_cat.I}, p-value: {moran_bv_lasso.p_sim}")



In [ ]:
from esda.moran import Moran_BV
from libpysal.weights import KNN

#knn weights
weights = KNN.from_dataframe(df_sp, k=12)
weights.transform = 'r'

print("Bivariate Moran's I:")
moran_college_absent = Moran_BV(df_sp['college_going_rate'], df_sp['ChronicAbsenteeismRate'],weights)
print(f"(college/ Chronic Absenteeism): {moran_college_absent.I}, p-value: {moran_college_absent.p_sim}")

moran_college_frpm = Moran_BV(df_sp['college_going_rate'], df_sp[ 'pct_frpm'],weights)
print(f"(college/ FRPM): {moran_college_frpm.I}, p-value: {moran_college_frpm.p_sim}")

moran_ap_lasso = Moran_BV(df_sp['AP_Count'], df_sp['lasso_pred'],weights)
print(f"(college/ AP Count): {moran_ap_lasso.I}, p-value: {moran_ap_lasso.p_sim}")
moran_college_math = Moran_BV(df_sp['college_going_rate'], df_sp['pct_passed_math'],weights)
print(f"(college / math): {moran_college_math.I}, p-value: {moran_college_math.p_sim}")


moran_frpm_absent = Moran_BV(df_sp['pct_frpm'], df_sp['ChronicAbsenteeismRate'],weights)
print(f"(FRPM /Chronic Absenteeism): {moran_frpm_absent.I}, p-value: {moran_frpm_absent.p_sim}")

moran_AP_frpm = Moran_BV(df_sp['AP_Count'], df_sp[ 'pct_frpm'],weights)
print(f"(AP/FRPM): {moran_AP_frpm.I}, p-value: {moran_AP_frpm.p_sim}")

moran_ap_absent = Moran_BV(df_sp['AP_Count'], df_sp['ChronicAbsenteeismRate'],weights)
print(f"(AP / absent): {moran_ap_absent.I}, p-value: {moran_ap_absent.p_sim}")

moran_frpm_math = Moran_BV(df_sp['pct_frpm'], df_sp['pct_passed_math'],weights)
print(f"(frpm / math): {moran_frpm_math.I}, p-value: {moran_frpm_math.p_sim}")

moran_ap_math = Moran_BV(df_sp['AP_Count'], df_sp['pct_passed_math'],weights)
print(f"(ap / math): {moran_ap_math.I}, p-value: {moran_ap_math.p_sim}")

moran_absent_math = Moran_BV(df_sp['ChronicAbsenteeismRate'], df_sp['pct_passed_math'],weights)
print(f"(absent / math): {moran_absent_math.I}, p-value: {moran_absent_math.p_sim}")

# can bring math test scores back in here


In [ ]:

shap_df = pd.DataFrame(
    shap_values_xgb,
    columns=[f'shap_{col}' for col in X_test.columns]
)
test_reset = test_df.reset_index(drop=True)
shap_df_reset = shap_df.reset_index(drop=True)
test_shap = pd.concat([test_reset, shap_df_reset], axis=1)

shap_sp_xgb = gpd.GeoDataFrame(test_shap,geometry=gpd.points_from_xy(test_shap['Longitude'], test_shap['Latitude']), crs='EPSG:4326')

# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")
# zi = (xi – min(x)) / (max(x) – min(x))
shap_sp_xgb[['college_going_rate',
             'ChronicAbsenteeismRate',
             'pct_frpm', 'AP_Count',
             "pct_passed_math"]] = shap_sp_xgb[['college_going_rate',
                                                     'ChronicAbsenteeismRate',
                                                     'pct_frpm', 'AP_Count',
                                                 "pct_passed_math"]].apply(lambda x: (x - x.min(skipna=True)) / (x.max(skipna=True) - x.min(skipna=True)))

import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 2, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.viridis  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate','ChronicAbsenteeismRate','pct_frpm',
    'AP_Count'],
    ['College Going Rate','Chronic Absenteeim',  '     FRPM',  '    AP Count']
):
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white',linewidth = 0.35)

    shap_sp_xgb.plot(column=col, ax=ax, markersize=5,
                     cmap=cmap, legend=False, alpha = 0.75)

    ax.set_axis_off()
    ax.set_title(title,loc='left')

cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])  # Position of the colorbar: [left, bottom, width, height]
norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Normalized range from 0 to 1
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Empty array for ScalarMappable
fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

plt.suptitle('Spatial Distribution of Key Features',x=0.45,y=0.93,fontweight='bold')

plt.subplots_adjust( wspace=-0.3, hspace=-0.05)

plt.tight_layout
plt.show()


In [ ]:

shap_df = pd.DataFrame(
    shap_values_xgb,
    columns=[f'shap_{col}' for col in X_test.columns]
)
test_reset = test_df.reset_index(drop=True)
shap_df_reset = shap_df.reset_index(drop=True)
test_shap = pd.concat([test_reset, shap_df_reset], axis=1)

shap_sp_xgb = gpd.GeoDataFrame(test_shap,geometry=gpd.points_from_xy(test_shap['Longitude'], test_shap['Latitude']), crs='EPSG:4326')

# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")


import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 2, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.viridis  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate','ChronicAbsenteeismRate','pct_frpm',
    'AP_Count'],
    ['College Going Rate','Chronic Absenteeim',  'FRPM',  'AP Count']
):
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white',linewidth = 0.35)

    shap_sp_xgb.plot(column=col, ax=ax, markersize=2, cmap=cmap, legend=False)

    ax.set_axis_off()
    ax.set_title(title)

# cax = fig.add_axes([0.91, 0.4, 0.015, 0.4])
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])
# fig.colorbar(sm, cax=cax, orientation='vertical', label='College Going Rate (%)', ticks=[0, 25, 50, 75, 100])
# plt.suptitle('Features',y=0.93,fontweight='bold')

plt.subplots_adjust( wspace=0, hspace=0)

plt.tight_layout
plt.show()


In [ ]:

shap_df = pd.DataFrame(
    shap_values_xgb,
    columns=[f'shap_{col}' for col in X_test.columns]
)
test_reset = test_df.reset_index(drop=True)
shap_df_reset = shap_df.reset_index(drop=True)
test_shap = pd.concat([test_reset, shap_df_reset], axis=1)

shap_sp_xgb = gpd.GeoDataFrame(test_shap,geometry=gpd.points_from_xy(test_shap['Longitude'], test_shap['Latitude']), crs='EPSG:4326')

# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")


import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 3, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.PiYG  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:6],
    ['ChronicAbsenteeismRate','pct_frpm',
    'AP_Count','shap_ChronicAbsenteeismRate', 'shap_pct_frpm','shap_AP_Count'],
    ['Chronic Absenteeim',  'FRPM',  'AP Count','Chronic Absenteeism SHAP','FRPM SHAP','AP County SHAP']
):
    ca_counties.plot(ax=ax, color='#333333', edgecolor='darkgrey',linewidth = 0.35)

    shap_sp_xgb.plot(column=col, ax=ax, markersize=2, cmap=cmap, legend=False)

    ax.set_axis_off()
    ax.set_title(title)

# cax = fig.add_axes([0.91, 0.4, 0.015, 0.4])
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])
# fig.colorbar(sm, cax=cax, orientation='vertical', label='College Going Rate (%)', ticks=[0, 25, 50, 75, 100])
plt.suptitle('XGBoost',y=0.92,fontweight='bold')

plt.subplots_adjust( wspace=0, hspace=0)

plt.tight_layout
plt.show()


In [ ]:

shap_df = pd.DataFrame(
    shap_values_rf,
    columns=[f'shap_{col}' for col in X_test.columns]
)
test_reset = test_df.reset_index(drop=True)
shap_df_reset = shap_df.reset_index(drop=True)
test_shap = pd.concat([test_reset, shap_df_reset], axis=1)

shap_sp_rf = gpd.GeoDataFrame(test_shap,geometry=gpd.points_from_xy(test_shap['Longitude'], test_shap['Latitude']), crs='EPSG:4326')

# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")


import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 3, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.PiYG  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:6],
    ['ChronicAbsenteeismRate','pct_frpm',
    'AP_Count','shap_ChronicAbsenteeismRate', 'shap_pct_frpm','shap_AP_Count'],
    ['Chronic Absenteeim',  'FRPM',  'AP Count','Chronic Absenteeism SHAP','FRPM SHAP','AP County SHAP']
):
    ca_counties.plot(ax=ax, color='#333333', edgecolor='darkgrey',linewidth = 0.35)

    shap_sp_rf.plot(column=col, ax=ax, markersize=2, cmap=cmap, legend=False)

    ax.set_axis_off()
    ax.set_title(title)

# cax = fig.add_axes([0.91, 0.4, 0.015, 0.4])
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])
# fig.colorbar(sm, cax=cax, orientation='vertical', label='College Going Rate (%)', ticks=[0, 25, 50, 75, 100])
plt.suptitle('Random Forest',y=0.92,fontweight='bold')

plt.subplots_adjust( wspace=0, hspace=0)

plt.tight_layout
plt.show()


In [ ]:

shap_df = pd.DataFrame(
    shap_values_lasso,
    columns=[f'shap_{col}' for col in X_test.columns]
)
test_reset = test_df.reset_index(drop=True)
shap_df_reset = shap_df.reset_index(drop=True)
test_shap = pd.concat([test_reset, shap_df_reset], axis=1)

shap_sp_lasso = gpd.GeoDataFrame(test_shap,geometry=gpd.points_from_xy(test_shap['Longitude'], test_shap['Latitude']), crs='EPSG:4326')

# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")


import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 3, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.PiYG  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:6],
    ['ChronicAbsenteeismRate','pct_frpm',
    'AP_Count','shap_ChronicAbsenteeismRate', 'shap_pct_frpm','shap_AP_Count'],
    ['Chronic Absenteeim',  'FRPM',  'AP Count','Chronic Absenteeism SHAP','FRPM SHAP','AP County SHAP']
):
    ca_counties.plot(ax=ax, color='#333333', edgecolor='darkgrey',linewidth = 0.35)

    shap_sp_lasso.plot(column=col, ax=ax, markersize=2, cmap=cmap, legend=False)

    ax.set_axis_off()
    ax.set_title(title)

# cax = fig.add_axes([0.91, 0.4, 0.015, 0.4])
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])
# fig.colorbar(sm, cax=cax, orientation='vertical', label='College Going Rate (%)', ticks=[0, 25, 50, 75, 100])
plt.suptitle('Lasso',y=0.92,fontweight='bold')

plt.subplots_adjust( wspace=0, hspace=0)

plt.tight_layout
plt.show()


In [ ]:

shap_df = pd.DataFrame(
    shap_values_cat,
    columns=[f'shap_{col}' for col in X_test.columns]
)
test_reset = test_df.reset_index(drop=True)
shap_df_reset = shap_df.reset_index(drop=True)
test_shap = pd.concat([test_reset, shap_df_reset], axis=1)

shap_sp_cat = gpd.GeoDataFrame(test_shap,geometry=gpd.points_from_xy(test_shap['Longitude'], test_shap['Latitude']), crs='EPSG:4326')

# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")


import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 3, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.PiYG  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:6],
    ['ChronicAbsenteeismRate','pct_frpm',
    'AP_Count','shap_ChronicAbsenteeismRate', 'shap_pct_frpm','shap_AP_Count'],
    ['Chronic Absenteeism',  'FRPM',  'AP Count','Chronic Absenteeism SHAP','FRPM SHAP','AP County SHAP']
):
    ca_counties.plot(ax=ax, color='#333333', edgecolor='darkgrey',linewidth = 0.35)

    shap_sp_cat.plot(column=col, ax=ax, markersize=2, cmap=cmap, legend=False)

    ax.set_axis_off()
    ax.set_title(title)

# cax = fig.add_axes([0.91, 0.4, 0.015, 0.4])
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])
# fig.colorbar(sm, cax=cax, orientation='vertical', label='College Going Rate (%)', ticks=[0, 25, 50, 75, 100])
plt.suptitle('Catboost',y=0.92,fontweight='bold')

plt.subplots_adjust( wspace=0, hspace=0)

plt.tight_layout
plt.show()


In [ ]:
bay_area_counties = ['Alameda', 'Contra Costa', 'Marin', 'Napa', 'San Francisco',
                     'San Mateo', 'Santa Clara', 'Solano', 'Sonoma']
la_county_name = ['Los Angeles']

ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
bay_area_counties_gdf = ca_counties[ca_counties['NAME'].isin(bay_area_counties)]
la_county_gdf = ca_counties[ca_counties['NAME'].isin(la_county_name)]

def plot_regional_features(gdf, data_gdf, title):
    features = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count']
    feature_titles = ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']

    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()
    cmap = mpl.cm.viridis


    for ax, col, title in zip(axes, features, feature_titles):
        gdf.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.35)

        data_gdf.plot(column=col, ax=ax, markersize=30,
                      cmap=cmap, legend=False, alpha=0.75,
                      edgecolor='black', linewidth=0.5)

        ax.set_title(title)
        ax.set_axis_off()

    # colorbar
    cax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

    plt.suptitle(f'{title} - Feature Distribution', y=0.95, fontweight='bold')
    plt.subplots_adjust(wspace=0.1, hspace=0.1)
    # plt.tight_layout()
    plt.show()

# bay
bay_area_shap = shap_sp_xgb[
    shap_sp_xgb['CountyName'].isin(bay_area_counties)
]
plot_regional_features(bay_area_counties_gdf, bay_area_shap, 'Bay Area')

# la county
la_county_shap = shap_sp_xgb[
    shap_sp_xgb['CountyName'].isin(la_county_name)
]

plot_regional_features(la_county_gdf, la_county_shap, 'Los Angeles County')

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# correlation matrix data
metrics = ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count', 'SBA Math Proficiency']
correlations = np.array([
    [1.000, -0.248, -0.393, 0.351, 0.385],
    [-0.248, 1.000, 0.212, -0.179, -0.259],
    [-0.393, 0.212, 1.000, -0.192, -0.478],
    [0.351, -0.179, -0.192, 1.000, 0.258],
    [0.385, -0.259, -0.478, 0.258, 1.000]
])
matrix = np.triu(correlations)
# Set up the matplotlib figure
plt.figure(figsize=(10, 8))

# Create heatmap using seaborn
sns.heatmap(correlations,
            xticklabels=metrics,
            yticklabels=metrics,
            cmap='RdBu',mask = matrix,
            vmin=-1,
            vmax=1,
            center=0, annot_kws={'size': 13},
            annot=True,
            fmt='.3f',
            square=True,
            cbar_kws={'label': "Bivariate Moran's I"})

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.title("Bivariate Moran's I in key School-Level Variables", pad=20)
plt.tight_layout()

plt.show()